In [ ]:
!pip install pandas numpy nltk textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.7 MB/s eta 0:00:00


In [ ]:
import nltk

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
import pandas as pd
import numpy as np
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import pos_tag
from textstat import textstat # สำหรับการนับพยางค์และความซับซ้อนของคำ
from google.colab import files



# กำหนดรายชื่อ Function Words, Prepositions, และ Pronouns
FUNCTION_WORDS = {'the', 'is', 'at', 'which', 'on', 'and', 'or', 'but', 'because'}
PREPOSITIONS = {'in', 'on', 'at', 'by', 'with'}
PRONOUN_TAGS = {'PRP', 'PRP$', 'WP', 'WP$'} # แท็ก NLTK สำหรับคำสรรพนาม (Personal, Possessive, Wh-pronouns)
LINKING_WORDS = {'but', 'and', 'or', 'because'} # คำที่ใช้ในการคำนวณ Sentence Complexity

def analyze_email_body(text):
    """
    คำนวณคุณลักษณะทางภาษาทั้งหมดจากข้อความอีเมล.
    """
    if pd.isna(text) or text is None:
        return (0,) * 19 # คืนค่า 0 ทั้งหมดถ้าข้อความเป็นค่าว่าง

    # 1. การเตรียมการ (Tokenization)
    text = str(text).lower() # แปลงเป็นตัวพิมพ์เล็กสำหรับวิเคราะห์คำส่วนใหญ่
    original_text = str(text) # เก็บข้อความดั้งเดิมไว้สำหรับบางการนับ

    # ใช้วิธี tokenize ของ NLTK สำหรับคำและประโยค
    words = word_tokenize(original_text)
    lower_words = [word.lower() for word in words if word.isalnum()] # กรองเอาเฉพาะคำที่เป็นตัวอักษร/ตัวเลข
    sentences = sent_tokenize(original_text)

    # คำนวณเบื้องต้น
    word_count = len(lower_words)
    char_count = len(original_text)
    sentence_count = len(sentences)

    # จัดการกรณีที่ word_count เป็น 0 เพื่อป้องกันการหารด้วยศูนย์
    if word_count == 0:
        return (0,) * 19


    # --- 1-7. Word Counts, Lengths, and Diversity ---

    # 3. Average Word Length
    avg_word_length = sum(len(word) for word in lower_words) / word_count

    # 5. Average Sentence Length
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0

    # 6. Unique Word Count
    unique_word_count = len(set(lower_words))

    # 7. Lexical Diversity
    lexical_diversity = unique_word_count / word_count


    # --- 8-10. Email, Uppercase Counts ---

    # 8. Number of Emails (นับคำที่มี @)
    email_count = sum(1 for word in words if '@' in word)

    # 9. Uppercase Word Count (นับคำที่เป็นตัวพิมพ์ใหญ่ทั้งหมดในคำเดิม)
    # ต้องใช้ original_text และ words ที่ยังไม่ได้แปลงเป็นตัวเล็ก
    uppercase_words = word_tokenize(str(text)) # Tokenize อีกครั้งเพื่อใช้คำเดิม
    upper_count = sum(1 for word in uppercase_words if word.isupper() and word.isalpha())

    # 10. Uppercase Word Count Ratio
    upper_ratio = upper_count / word_count


    # --- 11-12. Complexity Measures ---

    # 11. Complex Words Count (คำที่มีตัวอักษร > 6 ตัว)
    complex_count = sum(1 for word in lower_words if len(word) > 6)

    # 12. Average Syllables per Word (ใช้ textstat)
    try:
        total_syllables = sum(textstat.syllable_count(word) for word in lower_words)
        avg_syllables = total_syllables / word_count
    except:
        avg_syllables = 0


    # --- 13-14. Punctuation Counts ---

    # 13. Comma, Semicolon, and Colon Counts
    comma_count = original_text.count(',')
    semicolon_count = original_text.count(';')
    colon_count = original_text.count(':')

    # 14. Exclamation Count, Quotation Count, Dash Count
    exclamation_count = original_text.count('!')
    quotation_count = original_text.count('"')
    dash_count = original_text.count('-')


    # --- 15-19. Density and Ratio Measures ---

    # 15. Sentence Complexity Ratio (สัดส่วนของ but, and, or, because เทียบกับจำนวนคำทั้งหมด)
    linking_word_count = sum(1 for word in lower_words if word in LINKING_WORDS)
    sentence_complexity_ratio = linking_word_count / word_count

    # 16. Clause Density (สัดส่วนของ linking_word_count / sentence_count)
    clause_density = linking_word_count / sentence_count if sentence_count > 0 else 0

    # 17. Pronoun Density (NLTK POS Tagging)
    tagged_words = pos_tag(words) # ใช้คำที่ไม่ได้แปลงเป็นตัวเล็กในการ Tag
    pronoun_count = sum(1 for word, tag in tagged_words if tag in PRONOUN_TAGS)
    pronoun_density = pronoun_count / word_count

    # 18. Preposition Density
    preposition_count = sum(1 for word in lower_words if word in PREPOSITIONS)
    preposition_density = preposition_count / word_count

    # 19. Function Word Density
    function_word_count = sum(1 for word in lower_words if word in FUNCTION_WORDS)
    function_word_density = function_word_count / word_count


    # Return all 23 values
    return (
        word_count, char_count, avg_word_length, sentence_count, avg_sentence_length,
        unique_word_count, lexical_diversity, email_count, upper_count, upper_ratio,
        complex_count, avg_syllables,
        comma_count, semicolon_count, colon_count, exclamation_count, quotation_count, dash_count,
        sentence_complexity_ratio, clause_density, pronoun_density, preposition_density, function_word_density
    )

# กำหนดรายชื่อคอลัมน์ใหม่ตามลำดับที่ส่งคืนในฟังก์ชัน
NEW_COLUMNS = [
    'Word_Count', 'Character_Count', 'Average_Word_Length', 'Sentence_Count', 'Average_Sentence_Length',
    'Unique_Word_Count', 'Lexical_Diversity', 'Email_Count', 'Uppercase_Word_Count', 'Uppercase_Word_Count_Ratio',
    'Complex_Words_Count', 'Average_Syllables_per_Word',
    'Comma_Count', 'Semicolon_Count', 'Colon_Count', 'Exclamation_Count', 'Quotation_Count', 'Dash_Count',
    'Sentence_Complexity_Ratio', 'Clause_Density', 'Pronoun_Density', 'Preposition_Density', 'Function_Word_Density'
]

# 1. โหลดข้อมูล
try:
    df = pd.read_csv(myfile)
    print("โหลดข้อมูลเรียบร้อยแล้ว")
except FileNotFoundError:
    print(f"ไม่พบไฟล์ '{myfile}' กรุณาตรวจสอบชื่อไฟล์และพาธ")
    # สร้าง DataFrame ตัวอย่างสำหรับสาธิต
    data = {
        'Subject': ['Urgent action needed', 'Your account details', 'Holiday greetings'],
        'Body': [
            "Dear User, your account has been compromised. Click HERE now to verify. This is a very complex word. BUT and OR because",
            "Hi, this is a legitimate email. Please ignore the previous message. Contact us at support@real.com if you have any questions!",
            "Thank you! Our special offer expires soon. Visit our website: http://example.com"
        ],
        'Label': ['phishing', 'legitimate', 'legitimate']
    }
    df = pd.DataFrame(data)


# 2. ใช้ฟังก์ชัน apply เพื่อคำนวณคุณลักษณะ
# ฟังก์ชัน apply จะเรียกใช้ analyze_email_body สำหรับทุกแถวในคอลัมน์ 'Body'
# และเก็บผลลัพธ์เป็น Series ของ Tuples
new_features = df['Body'].apply(lambda x: pd.Series(analyze_email_body(x), index=NEW_COLUMNS))


# 3. รวมคุณลักษณะใหม่เข้ากับ DataFrame เดิม
df = pd.concat([df, new_features], axis=1)

# 4. แสดงผลลัพธ์
print("\n--- ตัวอย่าง DataFrame ที่มีคุณลักษณะใหม่ ---")
print(df[['Body'] + NEW_COLUMNS].head())

# 5. บันทึกไฟล์ที่วิเคราะห์แล้ว (ถ้าต้องการ)
df.to_csv('emails_with_features.csv', index=False)
files.download('emails_with_features.csv')

โหลดข้อมูลเรียบร้อยแล้ว

--- ตัวอย่าง DataFrame ที่มีคุณลักษณะใหม่ ---
                                                Body  Word_Count  \
0  Dear David,\n\nI hope this email finds you in ...       332.0   
1  Dear Julia,\n\nWe hope this email finds you we...       301.0   
2  Dear Toni, \n\nI hope this message finds you w...       283.0   
3  Dear Chelsea,\n\nI hope this communication fin...       328.0   
4  Dear Christine, \n\nI hope this email finds yo...       308.0   

   Character_Count  Average_Word_Length  Sentence_Count  \
0           2139.0             5.009036            18.0   
1           2011.0             4.900332            22.0   
2           1788.0             5.120141            16.0   
3           2218.0             5.548780            20.0   
4           2042.0             5.084416            17.0   

   Average_Sentence_Length  Unique_Word_Count  Lexical_Diversity  Email_Count  \
0                18.444444              172.0           0.518072          0.0   
1  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install pandas numpy nltk textstat
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
import pandas as pd
import numpy as np
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import pos_tag
from textstat import textstat # สำหรับการนับพยางค์และความซับซ้อนของคำ
from google.colab import files



# กำหนดรายชื่อ Function Words, Prepositions, และ Pronouns
FUNCTION_WORDS = {'the', 'is', 'at', 'which', 'on', 'and', 'or', 'but', 'because'}
PREPOSITIONS = {'in', 'on', 'at', 'by', 'with'}
PRONOUN_TAGS = {'PRP', 'PRP$', 'WP', 'WP$'} # แท็ก NLTK สำหรับคำสรรพนาม (Personal, Possessive, Wh-pronouns)
LINKING_WORDS = {'but', 'and', 'or', 'because'} # คำที่ใช้ในการคำนวณ Sentence Complexity

def analyze_email_body(text):
    """
    คำนวณคุณลักษณะทางภาษาทั้งหมดจากข้อความอีเมล.
    """
    if pd.isna(text) or text is None:
        return (0,) * 19 # คืนค่า 0 ทั้งหมดถ้าข้อความเป็นค่าว่าง

    # 1. การเตรียมการ (Tokenization)
    text = str(text).lower() # แปลงเป็นตัวพิมพ์เล็กสำหรับวิเคราะห์คำส่วนใหญ่
    original_text = str(text) # เก็บข้อความดั้งเดิมไว้สำหรับบางการนับ

    # ใช้วิธี tokenize ของ NLTK สำหรับคำและประโยค
    words = word_tokenize(original_text)
    lower_words = [word.lower() for word in words if word.isalnum()] # กรองเอาเฉพาะคำที่เป็นตัวอักษร/ตัวเลข
    sentences = sent_tokenize(original_text)

    # คำนวณเบื้องต้น
    word_count = len(lower_words)
    char_count = len(original_text)
    sentence_count = len(sentences)

    # จัดการกรณีที่ word_count เป็น 0 เพื่อป้องกันการหารด้วยศูนย์
    if word_count == 0:
        return (0,) * 19


    # --- 1-7. Word Counts, Lengths, and Diversity ---

    # 3. Average Word Length
    avg_word_length = sum(len(word) for word in lower_words) / word_count

    # 5. Average Sentence Length
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0

    # 6. Unique Word Count
    unique_word_count = len(set(lower_words))

    # 7. Lexical Diversity
    lexical_diversity = unique_word_count / word_count


    # --- 8-10. Email, Uppercase Counts ---

    # 8. Number of Emails (นับคำที่มี @)
    email_count = sum(1 for word in words if '@' in word)

    # 9. Uppercase Word Count (นับคำที่เป็นตัวพิมพ์ใหญ่ทั้งหมดในคำเดิม)
    # ต้องใช้ original_text และ words ที่ยังไม่ได้แปลงเป็นตัวเล็ก
    uppercase_words = word_tokenize(str(text)) # Tokenize อีกครั้งเพื่อใช้คำเดิม
    upper_count = sum(1 for word in uppercase_words if word.isupper() and word.isalpha())

    # 10. Uppercase Word Count Ratio
    upper_ratio = upper_count / word_count


    # --- 11-12. Complexity Measures ---

    # 11. Complex Words Count (คำที่มีตัวอักษร > 6 ตัว)
    complex_count = sum(1 for word in lower_words if len(word) > 6)

    # 12. Average Syllables per Word (ใช้ textstat)
    try:
        total_syllables = sum(textstat.syllable_count(word) for word in lower_words)
        avg_syllables = total_syllables / word_count
    except:
        avg_syllables = 0


    # --- 13-14. Punctuation Counts ---

    # 13. Comma, Semicolon, and Colon Counts
    comma_count = original_text.count(',')
    semicolon_count = original_text.count(';')
    colon_count = original_text.count(':')

    # 14. Exclamation Count, Quotation Count, Dash Count
    exclamation_count = original_text.count('!')
    quotation_count = original_text.count('"')
    dash_count = original_text.count('-')


    # --- 15-19. Density and Ratio Measures ---

    # 15. Sentence Complexity Ratio (สัดส่วนของ but, and, or, because เทียบกับจำนวนคำทั้งหมด)
    linking_word_count = sum(1 for word in lower_words if word in LINKING_WORDS)
    sentence_complexity_ratio = linking_word_count / word_count

    # 16. Clause Density (สัดส่วนของ linking_word_count / sentence_count)
    clause_density = linking_word_count / sentence_count if sentence_count > 0 else 0

    # 17. Pronoun Density (NLTK POS Tagging)
    tagged_words = pos_tag(words) # ใช้คำที่ไม่ได้แปลงเป็นตัวเล็กในการ Tag
    pronoun_count = sum(1 for word, tag in tagged_words if tag in PRONOUN_TAGS)
    pronoun_density = pronoun_count / word_count

    # 18. Preposition Density
    preposition_count = sum(1 for word in lower_words if word in PREPOSITIONS)
    preposition_density = preposition_count / word_count

    # 19. Function Word Density
    function_word_count = sum(1 for word in lower_words if word in FUNCTION_WORDS)
    function_word_density = function_word_count / word_count


    # Return all 23 values
    return (
        word_count, char_count, avg_word_length, sentence_count, avg_sentence_length,
        unique_word_count, lexical_diversity, email_count, upper_count, upper_ratio,
        complex_count, avg_syllables,
        comma_count, semicolon_count, colon_count, exclamation_count, quotation_count, dash_count,
        sentence_complexity_ratio, clause_density, pronoun_density, preposition_density, function_word_density
    )

# กำหนดรายชื่อคอลัมน์ใหม่ตามลำดับที่ส่งคืนในฟังก์ชัน
NEW_COLUMNS = [
    'Word_Count', 'Character_Count', 'Average_Word_Length', 'Sentence_Count', 'Average_Sentence_Length',
    'Unique_Word_Count', 'Lexical_Diversity', 'Email_Count', 'Uppercase_Word_Count', 'Uppercase_Word_Count_Ratio',
    'Complex_Words_Count', 'Average_Syllables_per_Word',
    'Comma_Count', 'Semicolon_Count', 'Colon_Count', 'Exclamation_Count', 'Quotation_Count', 'Dash_Count',
    'Sentence_Complexity_Ratio', 'Clause_Density', 'Pronoun_Density', 'Preposition_Density', 'Function_Word_Density'
]